# v7.0.0 — Multi-Frame Fusion Ablation + Open-Set LOSO + Statistical Analysis

**Notebook ini** menjalankan semua evaluasi deployment-oriented v7:
- **D3**: Ablation N×M frame matrix (N ∈ {1,3,5,10}, M ∈ {1,3,5,10}) → 16 konfigurasi
- **D4**: Fusion strategy ablation (mean, median, max, first) pada (N=5, M=5)
- **D5**: Latency profiling end-to-end per konfigurasi
- **A4**: Open-set LOSO evaluation (11 fold, 1 unknown per fold)
- **Gate-2**: Bootstrap CI, Cohen's d, Wilcoxon (informatif, bukan klaim primer)

**Prerequisite**: jalankan `v7_train_eval.ipynb` terlebih dahulu.
**Primary metric**: EER dengan multi-frame (5,5), d-prime, FRR@FAR=1% open-set.
**Klaim**: deployment-oriented, bukan statistik p < 0.05 (N=11 subjek, power tidak cukup).


## 1. Setup

In [ ]:
from google.colab import userdata
import os, sys, subprocess
from pathlib import Path

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL     = f'https://{GITHUB_TOKEN}@github.com/RZulfikri/Thesis.git'
REPO_DIR     = Path('/content/Thesis')
PROJECT_ROOT = REPO_DIR / '3DCNN'
COLAB_BRANCH = 'colab'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin

os.chdir(str(REPO_DIR))
subprocess.run(['git', 'checkout', COLAB_BRANCH], capture_output=True)
!git merge origin/main --no-edit --strategy-option theirs 2>/dev/null || true
!git config user.email 'colab-runner@thesis.local'
!git config user.name  'Colab Runner'

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))
print('Setup OK')

## 2. Konfigurasi

In [ ]:
import torch
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

SEEDS   = [42, 123, 2024, 7, 31337, 0, 1, 2, 3, 4]
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_DIR = PROJECT_ROOT / 'dataset'
RUNS_DIR = PROJECT_ROOT / 'runs' / 'v7_lowdata'
EVAL_DIR = PROJECT_ROOT / 'eval_results' / 'v7_lowdata'

# Direktori output analisis
import datetime
TS       = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
ANA_DIR  = PROJECT_ROOT / 'analysis' / f'v7_lowdata_{TS}'
ANA_DIR.mkdir(parents=True, exist_ok=True)

# Varian yang akan dianalisis (harus sudah ditraining di v7_train_eval.ipynb)
VARIANTS = [
    'standard',
    'arcface_m03',
    'arcface_m04',
    'arcface_m05',
    'arcface_s64',
    'cosface',
    'subcenter',
    'hybrid',
]

# Konfigurasi multi-frame ablation (D3)
N_LIST  = [1, 3, 5, 10]   # frames untuk enrollment
M_LIST  = [1, 3, 5, 10]   # frames untuk probe
N_BEST  = 5               # konfigurasi rekomendasi (D6)
M_BEST  = 5

N_POINTS = 1024           # titik per frame untuk inference (lebih kecil = lebih cepat)

print(f'Device: {DEVICE}')
print(f'ANA_DIR: {ANA_DIR}')
print(f'Variants: {VARIANTS}')

## 3. Helper: Load Model

In [ ]:
from models.siamese import SiamesePalmNet
from utils.normalizer import GeometryNormalizer

def load_model(variant, seed):
    ckpt_path  = RUNS_DIR / variant / f'seed_{seed}' / 'best.pth'
    norm_path  = RUNS_DIR / variant / f'seed_{seed}' / 'normalizer.json'
    if not ckpt_path.exists():
        return None, None

    model = SiamesePalmNet(
        geom_dim=13, use_geom=False, use_aux_loss=False,
        n_subjects=11, siamese_mode='concat',
    ).to(DEVICE)

    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    state = ckpt.get('model_state_dict', ckpt)
    # Migrasi legacy keys jika perlu
    if any(k.startswith('encoder.proj.') for k in state.keys()):
        migrated = {}
        for k, v in state.items():
            migrated[k.replace('encoder.proj.', 'encoder.proj_no_geom.')] = v
        state = migrated
    model.load_state_dict(state, strict=False)
    model.eval()

    normalizer = None
    if norm_path.exists():
        normalizer = GeometryNormalizer.load(str(norm_path))

    return model, normalizer

# Test load
v0, n0 = load_model(VARIANTS[0], SEEDS[0])
print(f'Model {VARIANTS[0]} seed={SEEDS[0]}: {"OK" if v0 is not None else "MISSING"}')

## 4. Session Splits

In [ ]:
from utils.dataset_lowdata import build_lowdata_splits_session_dirs

all_session_splits = build_lowdata_splits_session_dirs(str(DATA_DIR))

for split, subs in all_session_splits.items():
    total = sum(len(v) for v in subs.values())
    print(f'  {split:8s}: {len(subs)} subjek, {total} sesi')

print(f'\nSubjek: {sorted(all_session_splits["test"].keys())}')

## 5. D3 — Ablation N×M Frame Matrix

Jalankan multi-frame fusion dengan semua kombinasi (N, M) pada varian terbaik dari single-frame eval.

In [ ]:
from utils.eval_multiframe import eval_multiframe_ablation, ablation_to_dataframe

# Pilih varian untuk ablation N×M (gunakan semua, atau subset cepat)
ABLATION_VARIANTS = VARIANTS   # Bisa dikurangi: ['standard', 'arcface_m05', 'cosface']

ablation_results = {}   # {(variant, seed): ablation_dict}

for variant in ABLATION_VARIANTS:
    print(f'\n=== Ablation N×M: {variant} ===')
    variant_abl = []
    for seed in SEEDS:
        model, normalizer = load_model(variant, seed)
        if model is None:
            print(f'  seed={seed}: checkpoint missing, skip')
            continue

        res = eval_multiframe_ablation(
            model=model,
            enroll_session_splits=all_session_splits['train'],
            probe_session_splits=all_session_splits['test'],
            n_list=N_LIST,
            m_list=M_LIST,
            fusion_strategy='mean',
            device=DEVICE,
            n_points=N_POINTS,
            normalizer=normalizer,
            verbose=False,
        )
        ablation_results[(variant, seed)] = res
        # Ringkasan
        eer_55 = res.get((5, 5), {}).get('eer', float('nan'))
        eer_11 = res.get((1, 1), {}).get('eer', float('nan'))
        print(f'  seed={seed}: EER(1,1)={eer_11:.4f}  EER(5,5)={eer_55:.4f}')
        del model

print('\nAblation N×M selesai.')

In [ ]:
# Agregat: EER mean across seeds per (N, M) per variant
agg_ablation = {}   # {variant: DataFrame mean over seeds}

for variant in ABLATION_VARIANTS:
    rows = []
    for n in N_LIST:
        for m in M_LIST:
            eers = []
            for seed in SEEDS:
                res = ablation_results.get((variant, seed), {})
                eer = res.get((n, m), {}).get('eer', float('nan'))
                if not np.isnan(eer):
                    eers.append(eer)
            rows.append({'n_enroll': n, 'm_probe': m,
                         'eer_mean': np.mean(eers) if eers else float('nan'),
                         'eer_std':  np.std(eers)  if eers else float('nan')})
    agg_ablation[variant] = pd.DataFrame(rows)

# Tampilkan tabel untuk standard dan arcface_m05
for v in ['standard', 'arcface_m05']:
    if v in agg_ablation:
        pivot = agg_ablation[v].pivot(index='n_enroll', columns='m_probe', values='eer_mean')
        print(f'\n=== EER mean (10 seeds) — {v} ===')
        print(pivot.round(4).to_string())

In [ ]:
# Heatmap N×M untuk semua varian
n_v   = len(ABLATION_VARIANTS)
ncols = min(4, n_v)
nrows = (n_v + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = np.array(axes).flatten()

for i, variant in enumerate(ABLATION_VARIANTS):
    ax = axes[i]
    df = agg_ablation.get(variant)
    if df is None or df.empty:
        ax.set_title(f'{variant} — no data'); continue
    pivot = df.pivot(index='n_enroll', columns='m_probe', values='eer_mean')
    im = ax.imshow(pivot.values, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=0.15)
    ax.set_xticks(range(len(M_LIST))); ax.set_xticklabels(M_LIST)
    ax.set_yticks(range(len(N_LIST))); ax.set_yticklabels(N_LIST)
    ax.set_xlabel('M probe'); ax.set_ylabel('N enroll')
    ax.set_title(f'{variant}', fontsize=10)
    for r in range(len(N_LIST)):
        for c in range(len(M_LIST)):
            ax.text(c, r, f'{pivot.values[r,c]:.3f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('EER mean (10 seeds) — N×M ablation per variant', y=1.02)
plt.tight_layout()
plt.savefig(ANA_DIR / 'ablation_nm_heatmap.png', bbox_inches='tight')
plt.show()
print(f'Disimpan: {ANA_DIR}/ablation_nm_heatmap.png')

## 6. D4 — Fusion Strategy Ablation (N=5, M=5)

In [ ]:
from utils.eval_multiframe import fusion_strategy_ablation

FUSION_STRATEGIES = ['mean', 'median', 'max', 'first']
fusion_results = {}   # {(variant, seed, strategy): metrics}

for variant in ABLATION_VARIANTS:
    print(f'\n=== Fusion strategy: {variant} ===')
    for seed in SEEDS:
        model, normalizer = load_model(variant, seed)
        if model is None: continue

        res = fusion_strategy_ablation(
            model=model,
            enroll_session_splits=all_session_splits['train'],
            probe_session_splits=all_session_splits['test'],
            n_enroll=N_BEST, m_probe=M_BEST,
            strategies=FUSION_STRATEGIES,
            device=DEVICE, n_points=N_POINTS,
            normalizer=normalizer, verbose=False,
        )
        for strat, m in res.items():
            fusion_results[(variant, seed, strat)] = m
        del model

    # Ringkasan per strategy
    for strat in FUSION_STRATEGIES:
        eers = [fusion_results.get((variant, s, strat), {}).get('eer', float('nan')) for s in SEEDS]
        eers = [e for e in eers if not np.isnan(e)]
        if eers:
            print(f'  {strat:8s}: EER {np.mean(eers):.4f}±{np.std(eers):.4f}')

print('\nFusion strategy ablation selesai.')

## 7. D6 — Protokol Primer v7: Single vs Multi-Frame (N=5, M=5, best fusion)

In [ ]:
# Ambil fusion strategy terbaik dari D4 (default: mean)
BEST_FUSION = 'mean'   # Update jika D4 menunjukkan strategy lain lebih baik

# Load single-frame EER dari eval_results (v7_train_eval.ipynb)
results_sf = {}   # {(variant, seed): eer_single_frame}
results_mf = {}   # {(variant, seed): eer_multi_frame (5,5)}

for variant in VARIANTS:
    for seed in SEEDS:
        # Single-frame (dari evaluate.py)
        sf_f = EVAL_DIR / variant / f'seed_{seed}' / 'test' / 'results.json'
        if sf_f.exists():
            with open(sf_f) as fp:
                r = json.load(fp)
            results_sf[(variant, seed)] = r[0]['eer']

        # Multi-frame (dari ablation)
        abl = ablation_results.get((variant, seed), {})
        mf_eer = abl.get((N_BEST, M_BEST), {}).get('eer', float('nan'))
        if not np.isnan(mf_eer):
            results_mf[(variant, seed)] = mf_eer

# Tabel perbandingan
print(f'{'Variant':<18} {'SF EER (mean±std)':<22} {'MF EER N={N_BEST},M={M_BEST} (mean±std)':<28} {'Δ EER'}')
print('-' * 80)

for variant in VARIANTS:
    sf_eers = [results_sf.get((variant, s)) for s in SEEDS if results_sf.get((variant, s)) is not None]
    mf_eers = [results_mf.get((variant, s)) for s in SEEDS if results_mf.get((variant, s)) is not None]
    sf_str = f'{np.mean(sf_eers):.4f}±{np.std(sf_eers):.4f}' if sf_eers else 'missing'
    mf_str = f'{np.mean(mf_eers):.4f}±{np.std(mf_eers):.4f}' if mf_eers else 'missing'
    delta  = f'{np.mean(mf_eers) - np.mean(sf_eers):+.4f}' if (sf_eers and mf_eers) else 'N/A'
    print(f'{variant:<18} {sf_str:<22} {mf_str:<28} {delta}')

## 8. A4 — Open-Set LOSO Evaluation

In [ ]:
from utils.eval_openset import run_loso_eval, loso_to_dataframe

loso_results = {}   # {(variant, seed): loso_result}

for variant in VARIANTS:
    print(f'\n=== LOSO: {variant} ===')
    for seed in SEEDS:
        model, normalizer = load_model(variant, seed)
        if model is None: continue

        res = run_loso_eval(
            model=model,
            all_session_splits=all_session_splits,
            device=DEVICE,
            n_enroll=N_BEST, m_probe=M_BEST,
            fusion_strategy=BEST_FUSION,
            n_points=N_POINTS,
            normalizer=normalizer,
            split_name='test',
            verbose=False,
        )
        loso_results[(variant, seed)] = res
        s = res['summary']
        print(f'  seed={seed}: closed_EER={s["closed_set_eer_mean"]:.4f}  '
              f'FAR@unk={s["far_at_unknown_mean"]:.4f}  '
              f'FRR@FAR1%={s["frr_at_far1_mean"]:.4f}')
        del model

print('\nLOSO selesai.')

In [ ]:
# Tabel ringkasan LOSO per variant
loso_agg = []
for variant in VARIANTS:
    ceers, fars, frrs, dps = [], [], [], []
    for seed in SEEDS:
        r = loso_results.get((variant, seed))
        if r is None: continue
        s = r['summary']
        ceers.append(s['closed_set_eer_mean'])
        fars.append(s['far_at_unknown_mean'])
        frrs.append(s['frr_at_far1_mean'])
        dps.append(s['dprime_mean'])
    if ceers:
        loso_agg.append({
            'variant':       variant,
            'closed_EER':    f'{np.mean(ceers):.4f}±{np.std(ceers):.4f}',
            'FAR@unknown':   f'{np.mean(fars):.4f}±{np.std(fars):.4f}',
            'FRR@FAR=1%':    f'{np.mean(frrs):.4f}±{np.std(frrs):.4f}',
            'd-prime':       f'{np.mean(dps):.3f}±{np.std(dps):.3f}',
        })

df_loso = pd.DataFrame(loso_agg)
print('\n=== LOSO Summary (mean ± std across 10 seeds) ===')
print(df_loso.to_string(index=False))

## 8b. t-SNE — Multi-Frame Fused Embeddings (N=5, M=5)

t-SNE dari embedding yang sudah di-fuse (mean N=5 frame per sesi).
Dibandingkan dengan single-frame (N=1, M=1) untuk melihat apakah fusion memperjelas cluster.

In [ ]:
from sklearn.manifold import TSNE
from utils.eval_multiframe import _encode_frames, fuse_embeddings, _sample_n_frames

def extract_fused_embeddings(model, session_splits, n_frames, fusion_strategy='mean'):
    """Encode dan fuse n_frames per sesi, return embeddings + labels."""
    all_embs, all_labels = [], []
    for label, session_dirs in session_splits.items():
        for sdir in session_dirs:
            frame_dirs = _sample_n_frames(sdir, n_frames, seed=42)
            per_frame_embs = _encode_frames(model, frame_dirs, DEVICE, N_POINTS, normalizer=None)
            fused = fuse_embeddings(per_frame_embs, fusion_strategy)
            all_embs.append(fused)
            all_labels.append(label)
    return np.array(all_embs), all_labels

# Pilih top 4 varian untuk visualisasi
VIZ_VARIANTS = ['standard', 'arcface_m05', 'cosface', 'subcenter']
VIZ_SEED = 42

label_names = sorted(all_session_splits['test'].keys())
label_to_idx = {l: i for i, l in enumerate(label_names)}
cmap = plt.cm.get_cmap('tab20', len(label_names))

fig, axes = plt.subplots(len(VIZ_VARIANTS), 2, figsize=(10, 4.5*len(VIZ_VARIANTS)))

for row, variant in enumerate(VIZ_VARIANTS):
    model, normalizer = load_model(variant, VIZ_SEED)
    if model is None:
        for c in range(2):
            axes[row, c].set_title(f'{variant} — missing')
        continue

    for col, (n_fr, title_suffix) in enumerate([(1, 'Single-Frame (N=1)'), (N_BEST, f'Multi-Frame (N={N_BEST})')]):
        ax = axes[row, col]
        # Gabungkan test + holdout sessions
        combined = {}
        for sp in ['test', 'holdout']:
            for lab, dirs in all_session_splits[sp].items():
                combined.setdefault(lab, []).extend(dirs)

        embs, labels = extract_fused_embeddings(model, combined, n_fr)
        tsne = TSNE(n_components=2, perplexity=min(30, len(embs)-1), random_state=42)
        embs_2d = tsne.fit_transform(embs)
        lidx = [label_to_idx[l] for l in labels]

        ax.scatter(embs_2d[:, 0], embs_2d[:, 1], c=lidx, cmap=cmap, s=20, alpha=0.7, edgecolors='none')
        ax.set_title(f'{variant} — {title_suffix}', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])

    del model; torch.cuda.empty_cache()

handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=cmap(label_to_idx[l]),
           markersize=6, label=l) for l in label_names]
fig.legend(handles=handles, loc='lower center', ncol=min(6, len(label_names)),
           fontsize=8, bbox_to_anchor=(0.5, -0.01))
plt.suptitle('t-SNE: Single-Frame vs Multi-Frame Fused Embeddings (seed=42)', y=1.01)
plt.tight_layout()
plt.savefig(ANA_DIR / 'tsne_sf_vs_mf.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Disimpan: {ANA_DIR}/tsne_sf_vs_mf.png')

## 8c. Confusion Matrix — Single-Frame vs Multi-Frame (N=5, M=5)

Confusion matrix berdasarkan nearest-neighbor matching di embedding space.
Membandingkan akurasi identifikasi single-frame vs multi-frame fused.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def compute_nn_confusion_matrix(model, train_sessions, probe_sessions, n_frames, fusion_strategy='mean'):
    """Nearest-neighbor classification: fuse n_frames per sesi, match probe ke train terdekat."""
    # Encode gallery (train)
    gallery_embs, gallery_labels = extract_fused_embeddings(model, train_sessions, n_frames, fusion_strategy)
    # Encode probes (test + holdout)
    probe_embs, probe_labels = extract_fused_embeddings(model, probe_sessions, n_frames, fusion_strategy)

    if len(gallery_embs) == 0 or len(probe_embs) == 0:
        return None, [], []

    # Cosine similarity -> nearest neighbor
    g_norm = gallery_embs / (np.linalg.norm(gallery_embs, axis=1, keepdims=True) + 1e-9)
    p_norm = probe_embs / (np.linalg.norm(probe_embs, axis=1, keepdims=True) + 1e-9)
    sim = p_norm @ g_norm.T
    pred_idx = sim.argmax(axis=1)
    preds = [gallery_labels[i] for i in pred_idx]

    cm = confusion_matrix(probe_labels, preds, labels=label_names)
    return cm, probe_labels, preds

# Gabungkan test + holdout untuk probe
combined_probe = {}
for sp in ['test', 'holdout']:
    for lab, dirs in all_session_splits[sp].items():
        combined_probe.setdefault(lab, []).extend(dirs)

# Plot confusion matrix: single-frame (left) vs multi-frame (right) per varian
fig, axes = plt.subplots(len(VIZ_VARIANTS), 2, figsize=(11, 5*len(VIZ_VARIANTS)))

for row, variant in enumerate(VIZ_VARIANTS):
    model, normalizer = load_model(variant, VIZ_SEED)
    if model is None:
        for c in range(2):
            axes[row, c].set_title(f'{variant} — missing')
        continue

    for col, (n_fr, title_suffix) in enumerate([
        (1, 'Single-Frame (N=1)'),
        (N_BEST, f'Multi-Frame (N={N_BEST})')
    ]):
        ax = axes[row, col]
        cm, true_labs, pred_labs = compute_nn_confusion_matrix(
            model, all_session_splits['train'], combined_probe, n_fr
        )
        if cm is None:
            ax.set_title(f'{variant} — no data'); continue

        acc = np.trace(cm) / cm.sum() * 100
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
        disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False, xticks_rotation=45)
        ax.set_title(f'{variant} — {title_suffix}\n(acc={acc:.1f}%)', fontsize=10)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True' if col == 0 else '')

    del model; torch.cuda.empty_cache()

plt.suptitle(f'Confusion Matrix: Single vs Multi-Frame NN Classification (seed={VIZ_SEED})', y=1.01)
plt.tight_layout()
plt.savefig(ANA_DIR / 'confusion_matrix_sf_vs_mf.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Disimpan: {ANA_DIR}/confusion_matrix_sf_vs_mf.png')

## 9. Gate-2 — Effect Size + Bootstrap CI + Wilcoxon (informatif)

In [ ]:
from scipy import stats

def cohen_d(a, b):
    """Cohen's d untuk dua sampel berpasangan."""
    diff = np.array(a) - np.array(b)
    return float(np.mean(diff) / (np.std(diff) + 1e-9))

def bootstrap_ci(a, b, n_boot=2000, ci=0.95, seed=0):
    """Bootstrap CI untuk mean(a) - mean(b)."""
    rng  = np.random.default_rng(seed)
    diffs = []
    n = len(a)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        diffs.append(np.mean(np.array(a)[idx]) - np.mean(np.array(b)[idx]))
    lo, hi = np.percentile(diffs, [(1-ci)/2*100, (1+ci)/2*100])
    return lo, hi, np.mean(diffs)

# Bandingkan setiap varian vs 'standard' pada single-frame EER
BASELINE = 'standard'
baseline_sf = [results_sf.get((BASELINE, s)) for s in SEEDS if results_sf.get((BASELINE, s)) is not None]
baseline_mf = [results_mf.get((BASELINE, s)) for s in SEEDS if results_mf.get((BASELINE, s)) is not None]

print(f'=== Effect Size vs "{BASELINE}" (Test EER, n=10 seeds) ===')
print(f'{'Variant':<18} {'Δ SF EER':<12} {'d SF':<8} {'95% CI SF':<20} {'Δ MF EER':<12} {'d MF':<8} {'Wilcoxon p SF'}')
print('-' * 95)

gate2_passed = False
for variant in VARIANTS:
    if variant == BASELINE:
        print(f'{variant:<18} (baseline)')
        continue

    v_sf = [results_sf.get((variant, s)) for s in SEEDS if results_sf.get((variant, s)) is not None]
    v_mf = [results_mf.get((variant, s)) for s in SEEDS if results_mf.get((variant, s)) is not None]

    if not v_sf or not baseline_sf:
        print(f'{variant:<18} missing data'); continue

    n = min(len(v_sf), len(baseline_sf))
    d_sf  = cohen_d(v_sf[:n], baseline_sf[:n])
    lo, hi, delta_sf = bootstrap_ci(v_sf[:n], baseline_sf[:n])
    try:
        _, p_sf = stats.wilcoxon(v_sf[:n], baseline_sf[:n])
    except Exception:
        p_sf = float('nan')

    d_mf = float('nan'); delta_mf_str = 'N/A'
    if v_mf and baseline_mf:
        nm = min(len(v_mf), len(baseline_mf))
        d_mf = cohen_d(v_mf[:nm], baseline_mf[:nm])
        _, _, delta_mf = bootstrap_ci(v_mf[:nm], baseline_mf[:nm])
        delta_mf_str = f'{delta_mf:+.4f}'

    if abs(d_sf) >= 0.5 or abs(d_mf) >= 0.5:
        gate2_passed = True

    print(f'{variant:<18} {delta_sf:+.4f}      {d_sf:+.3f}   [{lo:+.4f},{hi:+.4f}]    '
          f'{delta_mf_str:<12} {d_mf:+.3f}   p={p_sf:.3f}')

print(f'\nGate-2 (|d| ≥ 0.5 pada minimal 1 varian): {"PASS" if gate2_passed else "FAIL (d kecil)"}')
print('Catatan: Wilcoxon p dilaporkan sebagai informasi sekunder (bukan klaim primer)')

## 10. D5 — Latency Budget

In [ ]:
import time
from utils.eval_multiframe import eval_multiframe

# Gunakan 1 model (variant=arcface_m05, seed=42) untuk latency profiling
LAT_VARIANT = 'arcface_m05'
LAT_SEED    = 42
model_lat, norm_lat = load_model(LAT_VARIANT, LAT_SEED)

latency_rows = []
if model_lat is not None:
    for n in N_LIST:
        for m in M_LIST:
            res = eval_multiframe(
                model=model_lat,
                enroll_session_splits=all_session_splits['train'],
                probe_session_splits=all_session_splits['test'],
                n_enroll=n, m_probe=m,
                fusion_strategy='mean',
                device=DEVICE, n_points=N_POINTS,
                normalizer=norm_lat,
            )
            latency_rows.append({
                'n_enroll': n, 'm_probe': m,
                'eer': res.get('eer', float('nan')),
                'lat_enroll_s': res.get('latency_enroll_s'),
                'lat_probe_s': res.get('latency_probe_s'),
            })
    del model_lat

df_lat = pd.DataFrame(latency_rows)
print('\n=== Latency Budget (s) — arcface_m05 seed=42 ===')
pivot_lat = df_lat.pivot(index='n_enroll', columns='m_probe', values='lat_probe_s')
print('Probe latency (s):'); print(pivot_lat.round(3).to_string())

# Identifikasi konfigurasi yang memenuhi target ≤ 1 detik
feasible = df_lat[df_lat['lat_probe_s'] <= 1.0][['n_enroll', 'm_probe', 'eer', 'lat_probe_s']]
print(f'\nKonfigurasi dengan latency probe ≤ 1 detik:')
print(feasible.sort_values('eer').to_string(index=False))

## 11. Simpan Hasil & Summary Report

In [ ]:
import json as _json

# Simpan dataframes
for variant in VARIANTS:
    if variant in agg_ablation:
        agg_ablation[variant].to_csv(ANA_DIR / f'ablation_nm_{variant}.csv', index=False)

df_loso.to_csv(ANA_DIR / 'loso_summary.csv', index=False)
df_lat.to_csv(ANA_DIR / 'latency.csv', index=False)

# Simpan agregat single-frame + multi-frame EER
agg_rows = []
for variant in VARIANTS:
    sf_eers = [results_sf.get((variant, s)) for s in SEEDS if results_sf.get((variant, s)) is not None]
    mf_eers = [results_mf.get((variant, s)) for s in SEEDS if results_mf.get((variant, s)) is not None]
    agg_rows.append({
        'variant': variant,
        'sf_eer_mean': np.mean(sf_eers) if sf_eers else float('nan'),
        'sf_eer_std':  np.std(sf_eers)  if sf_eers else float('nan'),
        'mf_eer_mean': np.mean(mf_eers) if mf_eers else float('nan'),
        'mf_eer_std':  np.std(mf_eers)  if mf_eers else float('nan'),
        'n_seeds_sf': len(sf_eers),
        'n_seeds_mf': len(mf_eers),
    })
pd.DataFrame(agg_rows).to_csv(ANA_DIR / 'aggregate_sf_mf.csv', index=False)

# Summary markdown
lines = [
    '# v7.0.0 Low-Data Regime — Summary',
    '',
    f'**Tanggal**: {TS}',
    f'**Setup**: 11 subjek × 15 sesi × 10 frame/sesi = 165 frame',
    f'**Protokol primer**: multi-frame fusion N={N_BEST}, M={M_BEST}, strategy={BEST_FUSION}',
    f'**Seeds**: {SEEDS}',
    '',
    '## Single-Frame vs Multi-Frame EER',
    '',
    '| Variant | SF EER | MF EER (5,5) | Δ |',
    '|---------|--------|--------------|---|',
]
for row in agg_rows:
    sf = f"{row['sf_eer_mean']:.4f}±{row['sf_eer_std']:.4f}" if not np.isnan(row['sf_eer_mean']) else 'N/A'
    mf = f"{row['mf_eer_mean']:.4f}±{row['mf_eer_std']:.4f}" if not np.isnan(row['mf_eer_mean']) else 'N/A'
    delta = f"{row['mf_eer_mean'] - row['sf_eer_mean']:+.4f}" if (not np.isnan(row['mf_eer_mean']) and not np.isnan(row['sf_eer_mean'])) else 'N/A'
    lines.append(f"| {row['variant']} | {sf} | {mf} | {delta} |")

lines += ['', '## LOSO Open-Set Summary', '']
lines.append(df_loso.to_markdown(index=False) if hasattr(df_loso, 'to_markdown') else df_loso.to_string())

lines += ['', f'## Gate-2: {"PASS" if gate2_passed else "FAIL"}']
lines += ['', '_Dihasilkan otomatis oleh v7_multiframe_compare.ipynb_']

summary_path = ANA_DIR / 'SUMMARY.md'
summary_path.write_text('\n'.join(lines))
print(f'Summary disimpan: {summary_path}')

In [ ]:
# Git save
import subprocess
def git_save(message, push=False):
    """Commit & push semua file (tanpa filter ukuran)."""
    os.chdir(str(REPO_DIR))
    subprocess.run(['git', 'add', '-A'], cwd=str(REPO_DIR))
    if subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=str(REPO_DIR)).returncode == 0:
        print('nothing to commit'); os.chdir(str(PROJECT_ROOT)); return
    ret = subprocess.run(['git', 'commit', '-m', message], cwd=str(REPO_DIR),
                         capture_output=True, text=True)
    if ret.returncode == 0:
        print(f'Committed: {message}')
    else:
        print(f'Commit error: {ret.stderr[:300]}')
    if push:
        ret = subprocess.run(['git', 'push', 'origin', COLAB_BRANCH],
                             cwd=str(REPO_DIR), capture_output=True, text=True)
        print('Pushed OK' if ret.returncode == 0 else f'Push error: {ret.stderr[:300]}')
    os.chdir(str(PROJECT_ROOT))

git_save('v7.1.0: multi-frame fusion + LOSO + analysis complete', push=True)
print('\nSelesai! Lihat', ANA_DIR)